# Cache Inspector

This notebook provides basic checks and statistics for all caches in the RAG system:

**Main Cache Database** (`cache.db`):
- `embedding_cache` - Vector embeddings indexed by text hash
- `llm_cache` - LLM outputs indexed by content hash
- `graph_cache` - Consistency graph instances
- `document_cache` - Document metadata and chunk information
- `bm25_index` - Inverted index for keyword search

**Academic Caches**:
- `academic_references.db` - Academic paper metadata cache
- `academic_citation_graph.db` - Citation graph database
- `academic_terminology.db` - Domain terminology database

In [ ]:
# Imports
import sqlite3
from pathlib import Path
import json
from datetime import datetime
import pandas as pd

## Configuration

In [ ]:
# Define paths
workspace_root = Path(__file__).resolve().parents[1] if '__file__' in globals() else Path.cwd().parents[0]
rag_data_path = workspace_root / 'rag_data'

# Cache database paths
MAIN_CACHE_DB = rag_data_path / 'cache.db'
ACADEMIC_REF_DB = rag_data_path / 'academic_references.db'
CITATION_GRAPH_DB = rag_data_path / 'academic_citation_graph.db'
TERMINOLOGY_DB = rag_data_path / 'academic_terminology.db'

print(f'RAG data path: {rag_data_path}')
print(f'Main cache: {MAIN_CACHE_DB}')
print(f'Academic refs: {ACADEMIC_REF_DB}')
print(f'Citation graph: {CITATION_GRAPH_DB}')
print(f'Terminology: {TERMINOLOGY_DB}')

## Helper Functions

In [ ]:
def get_db_size(db_path: Path) -> str:
    """Get human-readable database file size."""
    if not db_path.exists():
        return 'N/A'
    size_bytes = db_path.stat().st_size
    for unit in ['B', 'KB', 'MB', 'GB']:
        if size_bytes < 1024.0:
            return f'{size_bytes:.2f} {unit}'
        size_bytes /= 1024.0
    return f'{size_bytes:.2f} TB'

def get_table_stats(db_path: Path, table_name: str) -> dict:
    """Get basic statistics for a table."""
    if not db_path.exists():
        return {'error': 'Database does not exist'}
    
    try:
        conn = sqlite3.connect(str(db_path))
        cursor = conn.cursor()
        
        # Check if table exists
        cursor.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
            (table_name,)
        )
        if not cursor.fetchone():
            conn.close()
            return {'error': f'Table {table_name} does not exist'}
        
        # Get row count
        cursor.execute(f'SELECT COUNT(*) FROM {table_name}')
        row_count = cursor.fetchone()[0]
        
        # Get column info
        cursor.execute(f'PRAGMA table_info({table_name})')
        columns = [row[1] for row in cursor.fetchall()]
        
        conn.close()
        return {
            'row_count': row_count,
            'columns': columns
        }
    except Exception as e:
        return {'error': str(e)}

def list_all_tables(db_path: Path) -> list:
    """List all tables in a database."""
    if not db_path.exists():
        return []
    
    try:
        conn = sqlite3.connect(str(db_path))
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
        tables = [row[0] for row in cursor.fetchall()]
        conn.close()
        return tables
    except Exception as e:
        return [f'Error: {e}']

## Main Cache Database (cache.db)

In [ ]:
print('=' * 70)
print('MAIN CACHE DATABASE')
print('=' * 70)
print(f'Path: {MAIN_CACHE_DB}')
print(f'Size: {get_db_size(MAIN_CACHE_DB)}')
print(f'Exists: {MAIN_CACHE_DB.exists()}')
print()

if MAIN_CACHE_DB.exists():
    tables = list_all_tables(MAIN_CACHE_DB)
    print(f'Tables: {tables}')
    print()
    
    # Expected tables
    expected_tables = ['embedding_cache', 'llm_cache', 'graph_cache', 'document_cache', 'bm25_index']
    
    for table in expected_tables:
        stats = get_table_stats(MAIN_CACHE_DB, table)
        if 'error' in stats:
            print(f'{table}: {stats["error"]}')
        else:
            print(f'{table}:')
            print(f'  Rows: {stats["row_count"]:,}')
            print(f'  Columns: {len(stats["columns"])} ({stats["columns"][:5]}{"..." if len(stats["columns"]) > 5 else ""})')
        print()
else:
    print('Database does not exist. Run ingestion to create caches.')

### Sample Entries from Other Cache Tables

In [ ]:
if MAIN_CACHE_DB.exists():
    for table_name in ['document_cache', 'graph_cache', 'bm25_index']:
        try:
            conn = sqlite3.connect(str(MAIN_CACHE_DB))
            
            # Check if table exists
            cursor = conn.cursor()
            cursor.execute(
                "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
                (table_name,)
            )
            if not cursor.fetchone():
                print(f'{table_name}: Table does not exist')
                print()
                conn.close()
                continue
            
            # Read sample data
            df = pd.read_sql_query(f'SELECT * FROM {table_name} LIMIT 10', conn)
            conn.close()
            
            if not df.empty:
                print(f'{table_name} (top 10 rows):')
                # Truncate long columns for readability
                for col in df.columns:
                    if df[col].dtype == 'object':
                        df[col] = df[col].apply(
                            lambda x: str(x)[:100] + '...' if isinstance(x, str) and len(str(x)) > 100 else x
                        )
                print(df.to_string(index=False))
            else:
                print(f'{table_name}: No entries')
            print()
        except Exception as e:
            print(f'Error reading {table_name}: {e}')
            print()
else:
    print('Main cache database does not exist')

## Embedding Cache Details

In [ ]:
if MAIN_CACHE_DB.exists():
    try:
        conn = sqlite3.connect(str(MAIN_CACHE_DB))
        cursor = conn.cursor()
        
        # Check if table exists
        cursor.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name='embedding_cache'"
        )
        if cursor.fetchone():
            # Get hit statistics
            cursor.execute('SELECT COUNT(*), SUM(hits), AVG(hits), MAX(hits) FROM embedding_cache')
            count, total_hits, avg_hits, max_hits = cursor.fetchone()
            
            print('EMBEDDING CACHE STATISTICS')
            print('-' * 70)
            print(f'Total entries: {count:,}')
            print(f'Total hits: {int(total_hits or 0):,}')
            print(f'Average hits per entry: {avg_hits or 0:.2f}')
            print(f'Maximum hits: {int(max_hits or 0):,}')
            print()
            
            # Get top 10 most used embeddings
            cursor.execute(
                'SELECT text_hash, hits FROM embedding_cache ORDER BY hits DESC LIMIT 10'
            )
            top_embeddings = cursor.fetchall()
            
            if top_embeddings:
                print('Top 10 Most Used Embeddings:')
                for idx, (text_hash, hits) in enumerate(top_embeddings, 1):
                    print(f'  {idx}. {text_hash[:16]}... : {hits:,} hits')
        else:
            print('embedding_cache table does not exist')
        
        conn.close()
    except Exception as e:
        print(f'Error querying embedding cache: {e}')
else:
    print('Main cache database does not exist')

### Sample Embedding Cache Entries

In [ ]:
if MAIN_CACHE_DB.exists():
    try:
        conn = sqlite3.connect(str(MAIN_CACHE_DB))
        df = pd.read_sql_query('SELECT * FROM embedding_cache LIMIT 10', conn)
        conn.close()
        
        if not df.empty:
            print('Top 10 Embedding Cache Entries:')
            print(df.to_string(index=False))
        else:
            print('No entries in embedding_cache')
    except Exception as e:
        print(f'Error reading embedding_cache: {e}')
else:
    print('Main cache database does not exist')

## LLM Cache Details

In [ ]:
if MAIN_CACHE_DB.exists():
    try:
        conn = sqlite3.connect(str(MAIN_CACHE_DB))
        cursor = conn.cursor()
        
        # Check if table exists
        cursor.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name='llm_cache'"
        )
        if cursor.fetchone():
            # Get hit statistics
            cursor.execute('SELECT COUNT(*), SUM(hits), AVG(hits), MAX(hits) FROM llm_cache')
            count, total_hits, avg_hits, max_hits = cursor.fetchone()
            
            print('LLM CACHE STATISTICS')
            print('-' * 70)
            print(f'Total entries: {count:,}')
            print(f'Total hits: {int(total_hits or 0):,}')
            print(f'Average hits per entry: {avg_hits or 0:.2f}')
            print(f'Maximum hits: {int(max_hits or 0):,}')
            print()
            
            # Get model breakdown
            cursor.execute(
                'SELECT model, COUNT(*) as count FROM llm_cache GROUP BY model ORDER BY count DESC'
            )
            model_breakdown = cursor.fetchall()
            
            if model_breakdown:
                print('Breakdown by Model:')
                for model, count in model_breakdown:
                    print(f'  {model or "(unspecified)"}: {count:,} entries')
                print()
            
            # Get top 10 most used LLM calls
            cursor.execute(
                'SELECT content_hash, model, hits FROM llm_cache ORDER BY hits DESC LIMIT 10'
            )
            top_llm = cursor.fetchall()
            
            if top_llm:
                print('Top 10 Most Used LLM Calls:')
                for idx, (content_hash, model, hits) in enumerate(top_llm, 1):
                    print(f'  {idx}. {content_hash[:16]}... ({model or "unknown"}): {hits:,} hits')
        else:
            print('llm_cache table does not exist')
        
        conn.close()
    except Exception as e:
        print(f'Error querying LLM cache: {e}')
else:
    print('Main cache database does not exist')

### Sample LLM Cache Entries

In [ ]:
if MAIN_CACHE_DB.exists():
    try:
        conn = sqlite3.connect(str(MAIN_CACHE_DB))
        # Select key columns, truncate long fields for readability
        df = pd.read_sql_query('''
            SELECT content_hash, 
                   SUBSTR(prompt, 1, 50) as prompt_preview,
                   SUBSTR(response, 1, 50) as response_preview,
                   model, 
                   hits,
                   created_at
            FROM llm_cache 
            ORDER BY created_at DESC
            LIMIT 10
        ''', conn)
        conn.close()
        
        if not df.empty:
            print('Top 10 LLM Cache Entries (most recent):')
            print(df.to_string(index=False))
        else:
            print('No entries in llm_cache')
    except Exception as e:
        print(f'Error reading llm_cache: {e}')
else:
    print('Main cache database does not exist')

## Academic References Database

In [ ]:
print('=' * 70)
print('ACADEMIC REFERENCES DATABASE')
print('=' * 70)
print(f'Path: {ACADEMIC_REF_DB}')
print(f'Size: {get_db_size(ACADEMIC_REF_DB)}')
print(f'Exists: {ACADEMIC_REF_DB.exists()}')
print()

if ACADEMIC_REF_DB.exists():
    tables = list_all_tables(ACADEMIC_REF_DB)
    print(f'Tables: {tables}')
    print()
    
    for table in tables:
        if table.startswith('sqlite_'):
            continue
        stats = get_table_stats(ACADEMIC_REF_DB, table)
        if 'error' not in stats:
            print(f'{table}:')
            print(f'  Rows: {stats["row_count"]:,}')
            print(f'  Columns: {stats["columns"]}')
            print()
else:
    print('Database does not exist. Run academic ingestion to create.')

### Sample Academic Reference Entries

In [ ]:
if ACADEMIC_REF_DB.exists():
    tables = list_all_tables(ACADEMIC_REF_DB)
    for table in tables:
        if table.startswith('sqlite_'):
            continue
        try:
            conn = sqlite3.connect(str(ACADEMIC_REF_DB))
            df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 10', conn)
            conn.close()
            
            if not df.empty:
                print(f'{table} (top 10 rows):')
                # Truncate long columns
                for col in df.columns:
                    if df[col].dtype == 'object':
                        df[col] = df[col].apply(
                            lambda x: str(x)[:80] + '...' if isinstance(x, str) and len(str(x)) > 80 else x
                        )
                print(df.to_string(index=False))
            else:
                print(f'{table}: No entries')
            print()
        except Exception as e:
            print(f'Error reading {table}: {e}')
            print()
else:
    print('Academic references database does not exist')

## Citation Graph Database

In [ ]:
print('=' * 70)
print('CITATION GRAPH DATABASE')
print('=' * 70)
print(f'Path: {CITATION_GRAPH_DB}')
print(f'Size: {get_db_size(CITATION_GRAPH_DB)}')
print(f'Exists: {CITATION_GRAPH_DB.exists()}')
print()

if CITATION_GRAPH_DB.exists():
    tables = list_all_tables(CITATION_GRAPH_DB)
    print(f'Tables: {tables}')
    print()
    
    for table in tables:
        if table.startswith('sqlite_'):
            continue
        stats = get_table_stats(CITATION_GRAPH_DB, table)
        if 'error' not in stats:
            print(f'{table}:')
            print(f'  Rows: {stats["row_count"]:,}')
            print(f'  Columns: {stats["columns"]}')
            print()
    
    # Get citation statistics if applicable
    try:
        conn = sqlite3.connect(str(CITATION_GRAPH_DB))
        cursor = conn.cursor()
        
        # Check for common citation graph tables
        cursor.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name IN ('papers', 'citations')"
        )
        citation_tables = [row[0] for row in cursor.fetchall()]
        
        if 'papers' in citation_tables:
            cursor.execute('SELECT COUNT(*) FROM papers')
            paper_count = cursor.fetchone()[0]
            print(f'Total papers: {paper_count:,}')
        
        if 'citations' in citation_tables:
            cursor.execute('SELECT COUNT(*) FROM citations')
            citation_count = cursor.fetchone()[0]
            print(f'Total citations: {citation_count:,}')
        
        conn.close()
    except Exception as e:
        print(f'Error querying citation stats: {e}')
else:
    print('Database does not exist. Run academic ingestion with citation graph enabled.')

### Sample Citation Graph Entries

In [ ]:
if CITATION_GRAPH_DB.exists():
    tables = list_all_tables(CITATION_GRAPH_DB)
    for table in tables:
        if table.startswith('sqlite_'):
            continue
        try:
            conn = sqlite3.connect(str(CITATION_GRAPH_DB))
            df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 10', conn)
            conn.close()
            
            if not df.empty:
                print(f'{table} (top 10 rows):')
                # Truncate long columns
                for col in df.columns:
                    if df[col].dtype == 'object':
                        df[col] = df[col].apply(
                            lambda x: str(x)[:80] + '...' if isinstance(x, str) and len(str(x)) > 80 else x
                        )
                print(df.to_string(index=False))
            else:
                print(f'{table}: No entries')
            print()
        except Exception as e:
            print(f'Error reading {table}: {e}')
            print()
else:
    print('Citation graph database does not exist')

## Terminology Database

In [ ]:
print('=' * 70)
print('TERMINOLOGY DATABASE')
print('=' * 70)
print(f'Path: {TERMINOLOGY_DB}')
print(f'Size: {get_db_size(TERMINOLOGY_DB)}')
print(f'Exists: {TERMINOLOGY_DB.exists()}')
print()

if TERMINOLOGY_DB.exists():
    tables = list_all_tables(TERMINOLOGY_DB)
    print(f'Tables: {tables}')
    print()
    
    for table in tables:
        if table.startswith('sqlite_'):
            continue
        stats = get_table_stats(TERMINOLOGY_DB, table)
        if 'error' not in stats:
            print(f'{table}:')
            print(f'  Rows: {stats["row_count"]:,}')
            print(f'  Columns: {stats["columns"]}')
            print()
    
    # Get terminology statistics
    try:
        conn = sqlite3.connect(str(TERMINOLOGY_DB))
        cursor = conn.cursor()
        
        # Check for terms table
        cursor.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name='terms'"
        )
        if cursor.fetchone():
            cursor.execute('SELECT COUNT(*) FROM terms')
            term_count = cursor.fetchone()[0]
            print(f'Total terms: {term_count:,}')
            
            # Get top terms by frequency if frequency column exists
            cursor.execute('PRAGMA table_info(terms)')
            columns = [row[1] for row in cursor.fetchall()]
            
            if 'frequency' in columns:
                cursor.execute(
                    'SELECT term, frequency FROM terms ORDER BY frequency DESC LIMIT 10'
                )
                top_terms = cursor.fetchall()
                if top_terms:
                    print()
                    print('Top 10 Most Frequent Terms:')
                    for idx, (term, freq) in enumerate(top_terms, 1):
                        print(f'  {idx}. {term}: {freq:,} occurrences')
        
        conn.close()
    except Exception as e:
        print(f'Error querying terminology stats: {e}')
else:
    print('Database does not exist. Run academic ingestion with terminology extraction enabled.')

### Sample Terminology Entries

In [ ]:
if TERMINOLOGY_DB.exists():
    tables = list_all_tables(TERMINOLOGY_DB)
    for table in tables:
        if table.startswith('sqlite_'):
            continue
        try:
            conn = sqlite3.connect(str(TERMINOLOGY_DB))
            df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 10', conn)
            conn.close()
            
            if not df.empty:
                print(f'{table} (top 10 rows):')
                # Truncate long columns
                for col in df.columns:
                    if df[col].dtype == 'object':
                        df[col] = df[col].apply(
                            lambda x: str(x)[:80] + '...' if isinstance(x, str) and len(str(x)) > 80 else x
                        )
                print(df.to_string(index=False))
            else:
                print(f'{table}: No entries')
            print()
        except Exception as e:
            print(f'Error reading {table}: {e}')
            print()
else:
    print('Terminology database does not exist')

## Summary

In [ ]:
summary_data = []

# Main cache
if MAIN_CACHE_DB.exists():
    for table in ['embedding_cache', 'llm_cache', 'graph_cache', 'document_cache', 'bm25_index']:
        stats = get_table_stats(MAIN_CACHE_DB, table)
        summary_data.append({
            'Database': 'cache.db',
            'Table': table,
            'Rows': stats.get('row_count', 0) if 'error' not in stats else 0,
            'Status': 'OK' if 'error' not in stats else 'Missing'
        })

# Academic databases
for db_name, db_path in [
    ('academic_references.db', ACADEMIC_REF_DB),
    ('academic_citation_graph.db', CITATION_GRAPH_DB),
    ('academic_terminology.db', TERMINOLOGY_DB)
]:
    if db_path.exists():
        tables = list_all_tables(db_path)
        for table in tables:
            if table.startswith('sqlite_'):
                continue
            stats = get_table_stats(db_path, table)
            summary_data.append({
                'Database': db_name,
                'Table': table,
                'Rows': stats.get('row_count', 0) if 'error' not in stats else 0,
                'Status': 'OK' if 'error' not in stats else 'Error'
            })
    else:
        summary_data.append({
            'Database': db_name,
            'Table': 'N/A',
            'Rows': 0,
            'Status': 'Not Found'
        })

if summary_data:
    df_summary = pd.DataFrame(summary_data)
    print('=' * 70)
    print('CACHE SUMMARY')
    print('=' * 70)
    print(df_summary.to_string(index=False))
    print()
    print(f'Total cached entries: {df_summary["Rows"].sum():,}')
else:
    print('No cache databases found.')